**Step #0, Prompting the model for tables and saving them in csv form**



In [ ]:
%pip install transformers bitsandbytes accelerate

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
import os

folder_path = "/Qwen/a.LLM_Tables"

folder_path_LLM_statements = "/Qwen/b.LLM_Inferences"

folder_path_python_code = "/Qwen/c.checking_statements"

folder_path_python_output_checking_statements = "/Qwen/d.checking_statements_output"

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
import torch
print(torch.cuda.memory_summary())

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen3-32B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=quant_config,
    low_cpu_mem_usage=True,
)

In [ ]:
prompt0 = [
    {
        "role": "system",
        "content": "You are a careful researcher who generates clean, realistic synthetic statistical tables."
    },
    {
        "role": "user",
        "content": """Generate exactly one synthetic statistical table in valid CSV format.

Requirements:
1. The table must have between 5 and 10 columns.
2. The table must contain exactly 15 data rows, plus 1 header row.
3. Invent realistic column names. Include a mix of:
   - numeric columns (integers and/or floats)
   - Numeric columns must contain only plain numbers (no $, commas, or % symbols)
   - categorical columns
4. Make the data look plausible for a real-world dataset.
5. Ensure values are varied across rows.
6. All rows must have the same number of columns as the header.
7. Do not include commas inside cell values.
8. Do not include explanations, markdown, code fences, titles, or any text before or after the CSV.
9. Output only the CSV table.

The dataset can be about any realistic domain such as health, education, business, transportation, or demographics."""
    },
]

In [ ]:
from transformers import pipeline

prompt = tokenizer.apply_chat_template(
    prompt0,
    tokenize=False,
    add_generation_prompt=True
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

generation = generator(
    prompt,
    do_sample=False,
    max_new_tokens=5000,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)

print(generation[0]["generated_text"])

In [ ]:
import re

full_text = generation[0]["generated_text"]
full_text = re.sub(r"<think>.*?</think>", "", full_text, flags=re.DOTALL).strip()
print(full_text)

In [ ]:
import os
from io import StringIO
import csv

folder_path = "Qwen/a.LLM_Tables"
os.makedirs(folder_path, exist_ok=True)

# Assume full_text contains your LLM CSV output
candidate_tables = StringIO(full_text)
reader = csv.reader(candidate_tables)

# Read all rows
all_rows = [r for r in reader if any(cell.strip() for cell in r)]  # remove empty rows

if not all_rows:
    print("No rows found")
else:
    header = all_rows[0]       # first non-empty row is header
    rows = all_rows[1:]        # the rest are data rows

    # Ensure all rows have the same number of columns as header
    num_cols = len(header)
    valid_rows = [r for r in rows if len(r) == num_cols]

    # Combine header + valid rows for saving
    table_cleaned = [header] + valid_rows

    # Save CSV
    file_name = "table_1.csv"
    full_path = os.path.join(folder_path, file_name)
    with open(full_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerows(table_cleaned)

    print(f"Saved {file_name} with {len(valid_rows)} rows")
    print("Header:", header)
    print("First few rows:", valid_rows[:5])

**Step #1, Prompting the model for Statments**
INPUT READING and GENERATION OF STATEMENTS

---



In [ ]:

# Folder where the tables were saved
# folder_path = "/content/drive/MyDrive/RA2026/Qwen/Ayush/a.LLM_Tables/"

# List all CSV files in the folder
csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
csv_files.sort()  # optional: ensure consistent order

for csv_file in csv_files:
    full_path = os.path.join(folder_path, csv_file)

    # Read the table
    with open(full_path, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    if not lines:
        print(f"{csv_file} is empty, skipping")
        continue

    # Extract header and rows
    header = lines[0]
    rows = lines[1:]

    print(f"Processing {csv_file}: {len(rows)} rows")
    ##PROVIDE EXAMPLES OF WHAT KIND OF STATEMENTS TO GENERATE
    prompt1 = [
    {"role": "system", "content": "You are an expert data analyst and logician. You produce only TRUE, non-trivial, and high-information statements about tables."},
    {"role": "user", "content": f"""
Read the following table carefully:

{lines}

Your task is to generate exactly 15 logical statements that are TRUE of this table.

Definition:
A statement is INTERESTING if it reveals a non-obvious relationship, constraint, subgroup pattern, interaction between variables, or exception that cannot be seen from a single column alone.

Requirements:
1. Every statement must be factually true.
2. Each statement must reveal a meaningful pattern in the data.
3. Avoid trivial or obvious statements.
4. Do NOT restate individual rows.
5. Do NOT produce simple existence statements like:
   - "There exists a row where Country is X"
6. Do NOT produce weak majority statements like:
   - "Most rows have Age > 25"
7. Avoid statements that involve only ONE column.

Structure requirements:
8. At least 10 statements must involve at least TWO columns.
9. At least 5 statements must involve THREE or more columns.
10. At least 5 statements must use conditional (if-then) logic.
11. At least 3 statements must describe subgroup-specific patterns.
12. At least 2 statements must describe exceptions or constraints.

Preferred types of insights:
- Relationships between categories and numeric ranges
- Constraints linking multiple variables
- Patterns that only hold within specific subgroups
- Interactions (e.g., JobType + Department + Salary)
- Conditional dependencies across columns
- Non-obvious restrictions or correlations

Good examples:
- All rows in subgroup X with condition Y also satisfy Z.
- If a row has A and B, then C must also hold.
- Every high-value case belongs to a specific subset of categories.
- Rows with condition X only appear alongside condition Y.
- All exceptions to a broader trend belong to the same subgroup.

Bad examples:
- "There exists a row where City is New York"
- "Most rows have Salary > 50000"
- "All Finance rows have Salary > 60000" (unless it reveals something deeper)
- Any statement based on a single column only

Diversity:
13. Use a mix of:
    - universal statements
    - conditional statements
    - subgroup patterns
    - constrained patterns
14. Avoid repeating the same structure across all statements.

Output format:
15. Number statements 1 through 15.
16. Output only the statements, one per line.
17. Do NOT include explanations.

Final check:
Before outputting each statement, verify:
- it is true
- it involves multiple columns (unless absolutely necessary)
- it would be considered non-trivial by a human analyst

Return exactly 15 statements.
"""},]
    #Generate the statements necessary
    generation = generator(
    prompt1,
    do_sample=False,
    temperature=1.0,
    top_p=1,
    max_new_tokens=10000,
    eos_token_id=tokenizer.eos_token_id)
    print(f"Generation: {generation[0]['generated_text']}")
    # Get the assistant message from generated_text
    statements_LLM_output = generation[-1]['generated_text'][-1]  # last item
    clean_statements_LLM_output_text = statements_LLM_output['content']  # this is your CSV string
    statements_LLM_output_text = re.sub(r"<think>.*?</think>", "", clean_statements_LLM_output_text, flags=re.DOTALL)
    # Preview
    print(statements_LLM_output_text[1000:2000])
    #save the output of the LLM generated tasks
    file_name_LLM = f"LLM_statements_{csv_file[:-4]}.txt"

    folder_path_LLM_statements = "Qwen/b.LLM_Inferences"

    full_path_LLM_statements = os.path.join(folder_path_LLM_statements, file_name_LLM)


    with open(full_path_LLM_statements, "w", encoding="utf-8") as f:
      f.write(statements_LLM_output_text)
    print(f"Saved {file_name_LLM}.")


**STEP2: GENERATING A PYTHON CODE THAT WOULD CHECK TRUTH/FALSITY OF THE ABOVE STATEMENTS AND ALSO PROVIDE AN EXPLANATION FOR THAT**


In [ ]:
import re
import subprocess
import pandas as pd

# folder paths
folder_path_LLM_statements = "Qwen/b.LLM_Inferences"
folder_path               = "Qwen/a.LLM_Tables"
folder_path_python_code   = "Qwen/c.checking_statements"
folder_path_results = "Qwen/d.checking_statements_output"

# helper to extract
def extract_python_code(raw: str) -> str:
    """
    Strip markdown fences if present (```python ... ``` or ``` ... ```).
    Falls back to returning the full string if no fences are found.
    """
    # Match ```python ... ``` or ``` ... ``` (non-greedy, dotall)
    match = re.search(r"```(?:python)?\s*\n(.*?)```", raw, re.DOTALL)
    if match:
        return match.group(1).strip()
    # No fences found — return as-is (model already output plain code)
    return raw.strip()

LLM_statement_text_files = sorted(
    f for f in os.listdir(folder_path_LLM_statements) if f.endswith(".txt")
)

for LLM_statement_text_file in LLM_statement_text_files:
    full_path_stmt = os.path.join(folder_path_LLM_statements, LLM_statement_text_file)

    with open(full_path_stmt, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    if not lines:
        print(f"{LLM_statement_text_file} is empty, skipping")
        continue

    print(f"\nProcessing {LLM_statement_text_file}.")

    csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
    for csv_file in csv_files:
        if csv_file[:-4] not in LLM_statement_text_file:
            continue

        full_csv_path = os.path.join(folder_path, csv_file)
        df = pd.read_csv(full_csv_path)

        prompt2 = [
            {"role": "system", "content": "You are an expert data analyst."},
            {"role": "user", "content": f"""Do NOT repeat the instructions or the code provided.
Only output the requested Python code. Do NOT wrap the code in markdown fences.
Here's your task: Given the following statements {lines} and the header names of the table {df.head(0)}, write a python code (using pandas package)
that checks whether each statement is True or False and prints a justification.
The CSV is already located at: "{full_csv_path}" — hardcode this path directly in the script (no sys.argv).
It should also convert any turn numbers stored as strings into integers.
Everything you output must be valid, immediately runnable Python with no markdown or commentary outside comments.

Here is an example structure to follow:
import pandas as pd

def print_result(statement_no: int, description: str, truth: bool, explanation: str):
    status = "TRUE" if truth else "FALSE"
    print(f"\\nStatement {{statement_no}}: {{status}}")
    print(f"  - {{description}}")
    print(f"  - Explanation: {{explanation}}")

def stmt_1(df: pd.DataFrame):
    \"\"\"1. For all individuals, if the person is a woman, then her age is between 21 and 43.\"\"\"
    women = df[df["gender"] == "F"]
    condition = women["age"].between(21, 43, inclusive="both")
    truth = condition.all()
    if truth:
        expl = f"All {{len(women)}} women are aged 21–43."
    else:
        viol = women[~condition]
        expl = f"{{len(viol)}} women violate the rule (ages: {{', '.join(map(str, viol['age'].tolist()))}})."
    return truth, expl

def main():
    df = pd.read_csv("{full_csv_path}")
    checks = [(1, stmt_1)]  # extend for all statements
    for num, func in checks:
        truth, explanation = func(df)
        print_result(num, func.__doc__.strip(), truth, explanation)

if __name__ == "__main__":
    main()"""},
        ]

        # ── Generate ───────────────────────────────────────────────────────────
        generation = generator(
            prompt2,
            do_sample=False,
            temperature=1.0,
            top_p=1,
            max_new_tokens=100000,
            eos_token_id=tokenizer.eos_token_id,
        )

        python_LLM_output = generation[-1]["generated_text"][-1]
        raw_code = python_LLM_output["content"]

        # ── Clean: strip markdown fences reliably ──────────────────────────────
        python_code = extract_python_code(raw_code)

        # ── Save ───────────────────────────────────────────────────────────────
        python_file_name_LLM = f"python_code_{csv_file[:-4]}.py"
        full_path_py = os.path.join(folder_path_python_code, python_file_name_LLM)

        with open(full_path_py, "w", encoding="utf-8") as f:
            f.write(python_code)
        print(f"Saved {python_file_name_LLM}")

        # ── Execute automatically ──────────────────────────────────────────────
        print(f"Running {python_file_name_LLM}...")
        result = subprocess.run(
            ["python3", full_path_py],
            capture_output=True,
            text=True,
        )

        if result.stdout:
            print(result.stdout)
        if result.returncode != 0:
            print(f"[ERROR] Script exited with code {result.returncode}")
            print(result.stderr)
        else:
            print(f"[OK] {python_file_name_LLM} completed successfully.")

        results_file_name = f"validation_llama_inferences.txt"
        full_path_results_file = os.path.join(folder_path_results, results_file_name)

        with open(full_path_results_file, "w", encoding="utf-8") as f:
            f.write(result.stdout)
            if result.returncode != 0:
                f.write(f"\n[ERROR] Script exited with code {result.returncode}\n")
                f.write(result.stderr)

        print(f"Saved {results_file_name}")

In [ ]:
import pandas as pd

def print_result(statement_no: int, description: str, truth: bool, explanation: str):
    status = "TRUE" if truth else "FALSE"
    print(f"\nStatement {statement_no}: {status}")
    print(f"  - {description}")
    print(f"  - Explanation: {explanation}")

def stmt_1(df: pd.DataFrame):
    """1. All rows with ProductCategory "Groceries" have Sales exceeding $30,000 only when Region is "West" or "Southeast". """
    groceries = df[df['ProductCategory'] == 'Groceries']
    violating = groceries[(groceries['Sales'] > 30000) & (~groceries['Region'].isin(['West', 'Southeast']))]
    truth = violating.empty
    if truth:
        expl = f"All {len(groceries)} Groceries rows meet the condition."
    else:
        expl = f"{len(violating)} Groceries rows have Sales > $30,000 but Region not West/Southeast."
    return truth, expl

def stmt_2(df: pd.DataFrame):
    """2. Every row with PromotionType "BOGO" has ProfitMargin ≤ 30%. """
    bogo_rows = df[df['PromotionType'] == 'BOGO']
    violating = bogo_rows[bogo_rows['ProfitMargin'] > 30]
    truth = violating.empty
    if truth:
        expl = f"All {len(bogo_rows)} BOGO promotion rows have ProfitMargin ≤30%."
    else:
        expl = f"{len(violating)} BOGO rows have ProfitMargin >30%."
    return truth, expl

def stmt_3(df: pd.DataFrame):
    """3. In the "South" region, ProductCategory "Electronics" rows have higher CustomerRating (4.3) than "Home Goods" (4.0). """
    south_region = df[df['Region'] == 'South']
    electronics = south_region[south_region['ProductCategory'] == 'Electronics']
    home_goods = south_region[south_region['ProductCategory'] == 'Home Goods']
    if electronics.empty or home_goods.empty:
        truth = False
        expl = "No data for both categories in South region."
    else:
        e_avg = electronics['CustomerRating'].mean()
        h_avg = home_goods['CustomerRating'].mean()
        truth = e_avg > h_avg
        expl = f"Electronics average rating: {e_avg:.2f}, Home Goods: {h_avg:.2f}."
    return truth, expl

def stmt_4(df: pd.DataFrame):
    """4. For rows with StoreSize exceeding 3000 units, EmployeeCount is always above 20. """
    large_stores = df[df['StoreSize'] > 3000]
    violating = large_stores[large_stores['EmployeeCount'] <= 20]
    truth = violating.empty
    if truth:
        expl = f"All {len(large_stores)} large stores have EmployeeCount >20."
    else:
        expl = f"{len(violating)} large stores have EmployeeCount ≤20."
    return truth, expl

def stmt_5(df: pd.DataFrame):
    """5. All rows where Month is "December" have ProductCategory "Toys" and PromotionType "Holiday Promo". """
    dec_rows = df[df['Month'] == 'December']
    condition = (dec_rows['ProductCategory'] == 'Toys') & (dec_rows['PromotionType'] == 'Holiday Promo')
    truth = condition.all()
    if truth:
        expl = f"All {len(dec_rows)} December rows meet the criteria."
    else:
        viol = dec_rows[~condition]
        expl = f"{len(viol)} December rows do not have Toys and Holiday Promo."
    return truth, expl

def stmt_6(df: pd.DataFrame):
    """6. Rows in regions containing "North" with ProductCategory "Electronics" have Sales ≤ $25,000. """
    north_regions = df[df['Region'].str.contains('North', case=False)]
    electronics_north = north_regions[north_regions['ProductCategory'] == 'Electronics']
    violating = electronics_north[electronics_north['Sales'] > 25000]
    truth = violating.empty
    if truth:
        expl = f"All {len(electronics_north)} Electronics rows in North regions have Sales ≤$25,000."
    else:
        expl = f"{len(violating)} rows violate the condition."
    return truth, expl

def stmt_7(df: pd.DataFrame):
    """7. Rows with CustomerRating ≥ 4.5 have ProfitMargin ≥ 28%. """
    high_rating = df[df['CustomerRating'] >= 4.5]
    violating = high_rating[high_rating['ProfitMargin'] < 28]
    truth = violating.empty
    if truth:
        expl = f"All {len(high_rating)} high-rated rows have ProfitMargin ≥28%."
    else:
        expl = f"{len(violating)} rows have high rating but ProfitMargin <28%."
    return truth, expl

def stmt_8(df: pd.DataFrame):
    """8. All rows with Month "January" have ProductCategory "Electronics" or "Home Goods". """
    jan_rows = df[df['Month'] == 'January']
    condition = jan_rows['ProductCategory'].isin(['Electronics', 'Home Goods'])
    truth = condition.all()
    if truth:
        expl = f"All {len(jan_rows)} January rows have correct categories."
    else:
        viol = jan_rows[~condition]
        expl = f"{len(viol)} January rows have invalid categories."
    return truth, expl

def stmt_9(df: pd.DataFrame):
    """9. In regions with "South" in their name, ProductCategory "Apparel" rows have higher CustomerRating (4.6) than "Electronics" (4.3). """
    south_regions = df[df['Region'].str.contains('South', case=False)]
    apparel = south_regions[south_regions['ProductCategory'] == 'Apparel']
    electronics = south_regions[south_regions['ProductCategory'] == 'Electronics']
    if apparel.empty or electronics.empty:
        truth = False
        expl = "No data for both categories in South regions."
    else:
        a_avg = apparel['CustomerRating'].mean()
        e_avg = electronics['CustomerRating'].mean()
        truth = a_avg > e_avg
        expl = f"Apparel average: {a_avg:.2f}, Electronics: {e_avg:.2f}."
    return truth, expl

def stmt_10(df: pd.DataFrame):
    """10. If a row has ProductCategory "Electronics" and Region "Midwest", its ProfitMargin is strictly less than 25%. """
    midwest_electronics = df[(df['ProductCategory'] == 'Electronics') & (df['Region'] == 'Midwest')]
    violating = midwest_electronics[midwest_electronics['ProfitMargin'] >= 25]
    truth = violating.empty
    if truth:
        expl = f"All {len(midwest_electronics)} Midwest Electronics rows have ProfitMargin <25%."
    else:
        expl = f"{len(violating)} rows violate the condition."
    return truth, expl

def stmt_11(df: pd.DataFrame):
    """11. Rows with ProductCategory "Toys" are exclusively associated with Months "December" or "November". """
    toys_rows = df[df['ProductCategory'] == 'Toys']
    condition = toys_rows['Month'].isin(['December', 'November'])
    truth = condition.all()
    if truth:
        expl = f"All {len(toys_rows)} Toys rows are in December or November."
    else:
        viol = toys_rows[~condition]
        expl = f"{len(viol)} Toys rows are in invalid months."
    return truth, expl

def stmt_12(df: pd.DataFrame):
    """12. In regions with "South", rows with PromotionType "BOGO" have higher Sales ($14,500.75) than those with "Clearance" ($12,000.00). """
    south_regions = df[df['Region'].str.contains('South', case=False)]
    bogo_sales = south_regions[south_regions['PromotionType'] == 'BOGO']['Sales'].mean()
    clearance_sales = south_regions[south_regions['PromotionType'] == 'Clearance']['Sales'].mean()
    truth = (bogo_sales > clearance_sales)
    expl = f"BOGO average Sales: {bogo_sales:.2f}, Clearance: {clearance_sales:.2f}."
    return truth, expl

def stmt_13(df: pd.DataFrame):
    """13. For rows where Month is "March", ProductCategory is either "Groceries" or "Home Goods". """
    march_rows = df[df['Month'] == 'March']
    condition = march_rows['ProductCategory'].isin(['Groceries', 'Home Goods'])
    truth = condition.all()
    if truth:
        expl = f"All {len(march_rows)} March rows have valid categories."
    else:
        viol = march_rows[~condition]
        expl = f"{len(viol)} March rows have invalid categories."
    return truth, expl

def stmt_14(df: pd.DataFrame):
    """14. If a row has StoreSize between 1000–2000 units, EmployeeCount is always between 15–21. """
    mid_size = df[(df['StoreSize'] >= 1000) & (df['StoreSize'] <= 2000)]
    violating = mid_size[~mid_size['EmployeeCount'].between(15, 21, inclusive='both')]
    truth = violating.empty
    if truth:
        expl = f"All {len(mid_size)} mid-sized stores have EmployeeCount between 15-21."
    else:
        expl = f"{len(violating)} rows violate the condition."
    return truth, expl

def stmt_15(df: pd.DataFrame):
    """15. All rows with ProductCategory "Home Goods" and PromotionType "Clearance" have UnitsSold ≥ 400. """
    home_clearance = df[(df['ProductCategory'] == 'Home Goods') & (df['PromotionType'] == 'Clearance')]
    violating = home_clearance[home_clearance['UnitsSold'] < 400]
    truth = violating.empty
    if truth:
        expl = f"All {len(home_clearance)} Home Goods Clearance rows have UnitsSold ≥400."
    else:
        expl = f"{len(violating)} rows violate the condition."
    return truth, expl

def main():
    df = pd.read_csv("Qwen/a.LLM_Tables/table_1.csv")
    df['Sales'] = pd.to_numeric(df['Sales'].str.replace('$', '').str.replace(',', ''), errors='coerce')
    df['ProfitMargin'] = pd.to_numeric(df['ProfitMargin'].str.replace('%', ''), errors='coerce')
    df['CustomerRating'] = pd.to_numeric(df['CustomerRating'], errors='coerce')
    df['UnitsSold'] = pd.to_numeric(df['UnitsSold'], errors='coerce')
    df['EmployeeCount'] = pd.to_numeric(df['EmployeeCount'], errors='coerce')
    df['StoreSize'] = pd.to_numeric(df['StoreSize'], errors='coerce')
    
    checks = [
        (1, stmt_1),
        (2, stmt_2),
        (3, stmt_3),
        (4, stmt_4),
        (5, stmt_5),
        (6, stmt_6),
        (7, stmt_7),
        (8, stmt_8),
        (9, stmt_9),
        (10, stmt_10),
        (11, stmt_11),
        (12, stmt_12),
        (13, stmt_13),
        (14, stmt_14),
        (15, stmt_15),
    ]
    for num, func in checks:
        truth, explanation = func(df)
        print_result(num, func.__doc__.strip(), truth, explanation)

if __name__ == "__main__":
    main()